# Project Overview

## Geospatial Restaurant Recommendation Engine

**Objective:** To build a location-based recommendation system that suggests the top restaurants in Tallinn, Estonia, based on a user's exact GPS coordinates, budget constraints, and weighted review scores.

**Methodology:**
1. **Data Cleaning:** Handling missing URLs and standardizing data formats.
2. **Exploratory Data Analysis (EDA):** Geospatial mapping and analyzing categorical distributions.
3. **Feature Engineering:** Creating a weighted composite metric prioritizing Food Quality over Service and Atmosphere.
4. **Heuristic Engine:** Implementing the Haversine formula to calculate real-time shortest-path distances on the Earth's curved surface.

In [1]:
# Core Data Manipulation & Math
import numpy as np 
import pandas as pd 

# Data Visualization
import plotly.express as px

# Network & Utility Tools
import requests
from tqdm.notebook import tqdm

# Interactive UI & Deployment
import gradio as gr

In [2]:
df = pd.read_excel("/kaggle/input/datasets/stefannezgoda/tallinn-restaurants-2026/Tallinn_restaurants (2).xlsx")

# Data Overview

In [3]:
df.shape

(97, 10)

In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 97 entries, 0 to 96
Data columns (total 10 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Restaurant      97 non-null     object 
 1   URL             92 non-null     object 
 2   Address         97 non-null     object 
 3   Cuisine         97 non-null     object 
 4   Price_category  97 non-null     object 
 5   Atmosphere      97 non-null     float64
 6   Food            97 non-null     float64
 7   Service         97 non-null     float64
 8   Latitude        97 non-null     float64
 9   Longitude       97 non-null     float64
dtypes: float64(5), object(5)
memory usage: 7.7+ KB


In [5]:
df.describe()

,Atmosphere,Food,Service,Latitude,Longitude
count,97.000000,97.000000,97.000000,97.000000,97.000000
mean,4.209278,4.279381,4.169072,59.416731,24.788735
std,0.487144,0.425230,0.398006,0.150107,0.283567
min,2.400000,1.600000,2.600000,58.389640,24.572022
25%,4.000000,4.100000,4.000000,59.435863,24.743789
50%,4.300000,4.300000,4.200000,59.437705,24.747650
75%,4.600000,4.500000,4.400000,59.439840,24.758630
max,4.900000,4.900000,4.800000,59.493650,26.717700


In [6]:
df.head()

,Restaurant,URL,Address,Cuisine,Price_category,Atmosphere,Food,Service,Latitude,Longitude
0,Al Mare Grill,http://www.almarefood.ee/grill/,Mõisa tn 4,american,mid-range,3.5,3.0,3.1,59.427152,24.655658
1,Allee,https://www.alleerestoran.ee/,Kanuti 2,international,mid-range,4.0,4.1,3.9,59.440128,24.751131
2,Argentiina Lootsi,https://argentiina.ee/,Lootsi 8,argentinean,expensive,3.3,4.3,3.6,59.440838,24.765225
3,Argentiina Pärnu mnt,https://argentiina.ee/,Pärnu mnt 37,argentinean,expensive,4.1,4.5,4.2,59.429272,24.743789
4,Charlies Corner,https://charliescorner.ee/,Juhkentali 28,international,mid-range,3.9,3.9,3.9,59.427606,24.764699


In [7]:
df.duplicated().sum()

np.int64(0)

In [8]:
df.isnull().sum()

Restaurant        0
URL               5
Address           0
Cuisine           0
Price_category    0
Atmosphere        0
Food              0
Service           0
Latitude          0
Longitude         0
dtype: int64

In [9]:
df[df['URL'].isnull()]

,Restaurant,URL,Address,Cuisine,Price_category,Atmosphere,Food,Service,Latitude,Longitude
18,Rae,NaN,Raekoja plats 10,european,mid-range,4.0,4.3,4.3,59.437705,24.745503
29,FORK Resto,NaN,Veerenni 24,american,budget,4.5,4.5,4.5,59.424101,24.748084
53,III Draakon,NaN,Raeplats 1,estonian,budget,4.7,4.2,4.0,59.437323,24.745344
67,Burger Box,NaN,Kopli tn 4a,dutch,budget,4.6,4.5,3.8,59.441590,24.736560
85,Must Puudel,NaN,Kuninga tn 4,international,mid-range,4.6,4.3,4.3,59.436455,24.745533


In [10]:
# updating missing urls manually
df.loc[18, 'URL'] = 'https://www.raeharra.ee/en'
df.loc[29, 'URL'] = 'https://www.facebook.com/FORKresto'
df.loc[53, 'URL'] = 'https://www.kolmasdraakon.ee'
df.loc[67, 'URL'] = 'https://www.facebook.com/burgerboxbox/'
df.loc[85, 'URL'] = 'https://www.facebook.com/mustpuudel/'

print(df['URL'].isnull().sum())

0


In [11]:
# Checking URLs working or not
tqdm.pandas(desc="Checking URLs")

def test_url(url):
    if pd.isna(url):
        return "Missing"
    try:
        headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64)'}
        response = requests.get(url, headers=headers, timeout=5)
        if response.status_code == 200:
            return "Working"
        else:
            return f"Broken (Error {response.status_code})"
    except requests.RequestException:
        return "Broken (Connection Failed)"


df['URL_Status'] = df['URL'].progress_apply(test_url)

print(df['URL_Status'].value_counts())

broken_links = df[df['URL_Status'].str.contains("Broken", na=False)]
display(broken_links[['Restaurant', 'URL', 'URL_Status']])

Checking URLs:   0%|          | 0/97 [00:00<?, ?it/s]

URL_Status
Working    97
Name: count, dtype: int64


,Restaurant,URL,URL_Status


# Exploratory Data Analysis

In [12]:
fig = px.scatter_mapbox(
    df, 
    lat="Latitude", 
    lon="Longitude", 
    hover_name="Restaurant",
    hover_data=["Cuisine", "Price_category"], 
    color="Price_category",                 
    zoom=12, 
    height=600,
    title="Tallinn Restaurants by Price Category"
)

fig.update_layout(mapbox_style="open-street-map")

fig.show()

In [13]:
price_counts = df['Price_category'].value_counts().reset_index()
price_counts.columns = ['Price_category', 'Count']

fig_price = px.bar(
    price_counts, 
    x='Price_category', 
    y='Count', 
    title='Number of Restaurants by Price Category',
    color='Price_category',
    text='Count'
)
fig_price.show()

cuisine_counts = df['Cuisine'].value_counts().reset_index().head(10)
cuisine_counts.columns = ['Cuisine', 'Count']

fig_cuisine = px.bar(
    cuisine_counts, 
    x='Cuisine', 
    y='Count',
    title='Top 10 Cuisines in Tallinn',
    text='Count'
)
fig_cuisine.show()

# Feature Engineering

In [14]:
# Defining your weights
weight_food = 0.50
weight_service = 0.30
weight_atmosphere = 0.20

# Calculating the weighted rating
df['Overall_Rating'] = (
    (df['Food'] * weight_food) + 
    (df['Service'] * weight_service) + 
    (df['Atmosphere'] * weight_atmosphere)
).round(2)

# Checking the top 5 with the new weighted system
top_5_weighted = df[['Restaurant', 'Overall_Rating', 'Food', 'Service', 'Atmosphere']].sort_values(by='Overall_Rating', ascending=False).head()
display(top_5_weighted)

,Restaurant,Overall_Rating,Food,Service,Atmosphere
24,The Able Butcher,4.77,4.7,4.8,4.9
12,La Prima Pizza Vanalinn,4.75,4.7,4.8,4.8
44,180° by Matthias Diether,4.75,4.8,4.7,4.7
50,Vegan Restoran V,4.75,4.8,4.7,4.7
49,Rataskaevu 16,4.73,4.7,4.8,4.7


In [15]:
fig_rating = px.histogram(
    df, 
    x='Overall_Rating', 
    nbins=20, 
    title='Distribution of Overall Restaurant Ratings',
    color='Price_category',  
    marginal='box',          
    text_auto=True           
)

fig_rating.update_layout(bargap=0.1)
fig_rating.show()

## Distance Calculation (The Haversine Formula)

Because the Earth is a sphere, we cannot use standard straight-line (Euclidean) math to find the distance between two GPS coordinates. Instead, we use the **Haversine formula**, which calculates the shortest distance over the Earth's curved surface. We will vectorize this using `numpy` for maximum performance.

In [16]:
# Calculates the distance in kilometers between a user and a list of locations.
def calculate_distance(user_lat, user_lon, df_lats, df_lons):
    lat1, lon1, lat2, lon2 = map(np.radians, [user_lat, user_lon, df_lats, df_lons])
    
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat/2.0)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2.0)**2
    c = 2 * np.arcsin(np.sqrt(a))
    r = 6371 # radius in kms
    
    return np.round(c * r, 2)

# Test co-ordinates
mock_user_lat = 59.4370
mock_user_lon = 24.7535

# Calculates the distance to all restaurants and save it as a new column
df['Distance_km'] = calculate_distance(mock_user_lat, mock_user_lon, df['Latitude'], df['Longitude'])

# Show the 5 closest restaurants to the user
closest_spots = df[['Restaurant', 'Distance_km', 'Overall_Rating', 'Price_category']].sort_values(by='Distance_km').head()
display(closest_spots)

,Restaurant,Distance_km,Overall_Rating,Price_category
57,Maikrahv,0.00,3.77,expensive
75,Amalfi,0.08,4.35,mid-range
26,Beer Garden,0.17,4.06,mid-range
16,Platz,0.21,4.02,mid-range
81,Frank,0.25,4.07,mid-range


### Example Output

Before wrapping this logic into a user interface, let's look at a sample output. If a user is standing in the center of the Old Town (Lat: 59.4370, Lon: 24.7535) and wants **mid-range** dining, the engine instantly calculates the Haversine distances, ranks the options by their weighted scores, and returns the top 5 closest and best-rated results:

In [17]:
# Returns the top N recommended restaurants based on user location, budget, and weighted ratings.
def recommend_restaurants(user_lat, user_lon, budget_preference, top_n=10):
    filtered_df = df[df['Price_category'] == budget_preference.lower()].copy()
    if filtered_df.empty:
        return "No restaurants found for this budget category."
    filtered_df['Distance_km'] = calculate_distance(user_lat, user_lon, filtered_df['Latitude'], filtered_df['Longitude'])
    final_recommendations = filtered_df.sort_values(by=['Overall_Rating', 'Distance_km'], ascending=[False, True])
    
    return final_recommendations[['Restaurant', 'Cuisine', 'Overall_Rating', 'Distance_km', 'URL']].head(top_n)

# Testing function
test_recs = recommend_restaurants(59.4370, 24.7535, 'mid-range', top_n=5)
display(test_recs)

,Restaurant,Cuisine,Overall_Rating,Distance_km,URL
12,La Prima Pizza Vanalinn,italian,4.75,0.61,https://laprima.restaurant/vanalinn
50,Vegan Restoran V,european,4.75,0.61,https://veganrestoran.ee
49,Rataskaevu 16,international,4.73,0.63,https://rataskaevu16.ee
9,IO restaurant,italian,4.68,1.62,https://bacio.ee/menuu/
68,Grenka,european,4.65,1.58,https://grenka.ee


# Interactive Deployment (Gradio)

To make this algorithm accessible to end-users, the underlying logic has been wrapped in a Gradio UI. The interface allows users to input their current coordinates and budget, instantly triggering the backend Haversine calculations and returning the sorted Top 10 recommendations.

In [18]:
demo = gr.Interface(
    fn=recommend_restaurants,
    inputs=[
        gr.Number(value=59.4370, label="Your Latitude"),
        gr.Number(value=24.7535, label="Your Longitude"),
        gr.Dropdown(choices=['budget', 'mid-range', 'expensive', 'premium'], value='mid-range', label="Your Budget")
    ],
    outputs=gr.Dataframe(headers=['Restaurant', 'Cuisine', 'Rating', 'Distance (km)', 'URL']),
    title="Tallinn Restaurant Recommender",
    description="Enter your coordinates and budget to find the best-rated spots near you!",
    
    examples=[
        # Example 1: Town square, cheap
        [59.4370, 24.7535, "budget"],
        
        # Example 2: Near the port, fancy
        [59.4408, 24.7652, "expensive"],   
        
        # Example 3: Further out, mid-range
        [59.4271, 24.6556, "mid-range"]    
    ]
)

demo.launch(
    inline=True, 
    share=True,      
    height=800        
)

* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://acd41c2acebd3c3fe7.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


# Conclusion & Future Work

This notebook demonstrates a highly efficient, deterministic, rule-based recommendation system. Because it relies on strict mathematical heuristics rather than complex model inference, it is incredibly fast and perfectly accurate for localized searches.

**Future Enhancements (Machine Learning):**

* If this dataset were expanded to include historical user-review logs, the next iteration of this project would involve transitioning from a rule-based engine to an AI-driven one. Specifically, we could implement **Collaborative Filtering** (via Matrix Factorization) to predict user preferences based on similar users' behavioral patterns, or integrate a **Retrieval-Augmented Generation (RAG)** chatbot to allow users to interact with the database via natural language processing.